In [1]:
from transformers import AutoModel

model = AutoModel.from_pretrained("bert-base-cased")

/home/yang/projects/nn-zero-to-hero/hf-intro/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1951.37it/s]
[transformers] BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be 

In [2]:
model.save_pretrained("saved_model")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.46it/s]


In [3]:
model = AutoModel.from_pretrained("saved_model")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1922.23it/s]


In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

encoded_input = tokenizer("Hello, I'm a single sentence!")
print(encoded_input)

{'input_ids': [101, 8667, 117, 146, 112, 182, 170, 1423, 5650, 106, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


## demo.py — pipeline 版本

最高层 `pipeline` 一行搞定，然后拆开看内部：tokenizer → model → logits → softmax。

In [5]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import torch

# 1) 最高层：pipeline
clf = pipeline("sentiment-analysis")
print(clf(["I love this course!", "This is terrible."]))

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 1959.29it/s]


[{'label': 'POSITIVE', 'score': 0.9998835325241089}, {'label': 'NEGATIVE', 'score': 0.9996345043182373}]


In [6]:
# 2) 拆开 pipeline：手动 tokenizer -> model -> logits -> softmax
name = "distilbert-base-uncased-finetuned-sst-2-english"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForSequenceClassification.from_pretrained(name)
inputs = tok(["I love this course!", "This is terrible."],
             padding=True, truncation=True, return_tensors="pt")
print(inputs)                       # 看 input_ids / attention_mask
with torch.no_grad():
    logits = model(**inputs).logits
probs = torch.softmax(logits, dim=-1)
print(probs, model.config.id2label)  # 确认和 pipeline 输出一致

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 1904.14it/s]

{'input_ids': tensor([[ 101, 1045, 2293, 2023, 2607,  999,  102],
        [ 101, 2023, 2003, 6659, 1012,  102,    0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 0]])}
tensor([[1.1646e-04, 9.9988e-01],
        [9.9963e-01, 3.6548e-04]]) {0: 'NEGATIVE', 1: 'POSITIVE'}
